# Colab Runner: Swin SSL + Segmentation

This notebook runs your pipeline directly on Google Drive data.

It supports:
- quick smoke tests
- short sanity training runs
- full runs once configuration is verified

In [ ]:
# 1) Install dependencies
!pip -q install --upgrade transformers tqdm

In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Configure repo path and dataset paths
from pathlib import Path

# If your repo is in Drive, update this path if needed.
REPO_ROOT = Path('/content/drive/My Drive/Payne_lab_swin_transformer')

# Unlabeled data root used by SSL script (it recursively scans the 3 configured subfolders)
GDRIVE_ROOT = '/content/drive/My Drive/Petrographic images_ML work'

# Labeled image/mask dirs used by segmentation script
IMG_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/img'
MASK_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/masks_machine'

# Output folder for checkpoints/visualizations
OUT_ROOT = '/content/drive/My Drive/Petrographic images_ML work/model_outputs_221'

print('Repo exists:', REPO_ROOT.exists())
print('IMG dir exists:', Path(IMG_DIR).exists())
print('MASK dir exists:', Path(MASK_DIR).exists())

In [ ]:
# 4) Smoke tests (recommended first)
import os

os.chdir(REPO_ROOT)

# SSL smoke test: 1 epoch, 2 steps
!python code/model_training_pipeline/swin_ssl_pretrain_221.py \
  --gdrive_root "$GDRIVE_ROOT" \
  --epochs 1 \
  --batch_size 2 \
  --num_workers 2 \
  --max_steps_per_epoch 2 \
  --output_dir "$OUT_ROOT/ssl_smoke" \
  --amp

# Segmentation dataloader smoke test
!python code/model_training_pipeline/swin_training_pipeline_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --no_train

In [ ]:
# 5) Short SSL sanity run with reconstruction previews (uncomment to run)
# !python code/model_training_pipeline/swin_ssl_pretrain_221.py \
#   --gdrive_root "$GDRIVE_ROOT" \
#   --epochs 5 \
#   --batch_size 4 \
#   --num_workers 2 \
#   --crop 512 \
#   --mask_patch 16 \
#   --mask_ratio 0.55 \
#   --save_recon_every 1 \
#   --num_recon_samples 2 \
#   --output_dir "$OUT_ROOT/ssl_run" \
#   --amp

In [ ]:
# 6) Short segmentation sanity run (uncomment to run)
# !python code/model_training_pipeline/swin_training_pipeline_221.py \
#   --img_dir "$IMG_DIR" \
#   --mask_dir "$MASK_DIR" \
#   --epochs 1 \
#   --batch_size 1 \
#   --crop 512 \
#   --output_dir "$OUT_ROOT/seg_sanity"

In [ ]:
# 7) Show latest SSL reconstruction preview
from pathlib import Path
from IPython.display import Image as IPyImage, display

preview_dir = Path(f"{OUT_ROOT}/ssl_run/recon_previews")
if preview_dir.exists():
    preview_files = sorted(preview_dir.glob("epoch_*.png"))
    if preview_files:
        print("Latest preview:", preview_files[-1])
        display(IPyImage(filename=str(preview_files[-1])))
    else:
        print("No preview images found yet. Run cell 5 first.")
else:
    print("Preview directory not found. Run cell 5 first.")